# Constraint Satisfaction Problem
## Labor- und Präsentationsplanung als CSP

### Problemstellung

An einer Hochschule soll ein Präsentationsplan für eine Projektwoche erstellt werden. Jede Projektgruppe muss drei Präsentationen (A, B, C) durchführen, die jeweils einem Tag, Zeitslot, Raum und einer Prüfungskommission zugeordnet werden müssen. Die Planung unterliegt einer Reihe harter Constraints (z.B. keine Doppelbelegung von Räumen, Einhaltung von Kompetenzprofilen) sowie weichen Constraints (z.B. minimale Wartezeiten für Kommissionen).

Dieses Problem lässt sich formal als **Constraint Satisfaction Problem (CSP)** modellieren:

$$\text{CSP} = (X, D, C)$$

wobei $X$ die Menge der Variablen, $D$ die zugehörigen Domänen und $C$ die Menge der Constraints ist. Ziel ist es, jeder Variable $X_i$ einen Wert aus ihrer Domäne $D_i$ so zuzuweisen, dass alle Constraints in $C$ gleichzeitig erfüllt sind.


---
## 1. Konfiguration laden und analysieren

Die Konfigurationsdatei definiert alle Eingabeparameter des Problems: Gruppen, Tage, Zeitslots, Räume, Kommissionen mit ihren Themenkompetenzen sowie deren zeitliche Verfügbarkeiten. Drei Konfigurationen stehen zur Verfügung:

| Konfiguration | Gruppen | Kommissionen | Erwartetes Ergebnis |
|---|---|---|---|
| `configB.json` | 3 | 3 | Lösbar — kleines Entwicklungsproblem |
| `configA.json` | 6 | 6 | Lösbar — Hauptkonfiguration |
| `configC.json` | 9 | 3 | Potenziell unlösbar — Stresstest |


In [1]:
from csp_utils import analyze_and_display

In [2]:
# Aktive Konfiguration wählen — zum Testen nacheinander austauschen
config = analyze_and_display("configA.json")

# CSP-Konfiguration

Verwendete Datei: C:\Users\flore\Documents\GitHub\grundlagen-ki\constraints\C4\C4\configA.json
Lademodus: json


## Übersicht

Gruppen:
  G1, G2, G3, G4, G5, G6
Tage:
  Mon, Tue, Wed, Thu, Fri
Zeitslots:
  1: 08:00–10:00
  2: 10:30–12:30
  3: 13:00–14:30
  4: 15:00–17:00
Räume:
  L1, L2, L3
Anzahl Kommissionen:
  6
Anzahl Verfügbarkeits-Einträge:
  86


## Kompetenzen der Kommissionen

,Kommission,Themen,A,B,C
0,K1,B,,✓,
1,K2,"B, C",,✓,✓
2,K3,A,✓,,
3,K4,"A, B",✓,✓,
4,K5,C,,,✓
5,K6,"A, C",✓,,✓


## Legende

Kommission,Farbe
K1,
K2,
K3,
K4,
K5,
K6,


## Wochenplan der Verfügbarkeiten

,Mon,Tue,Wed,Thu,Fri
Zeitslot,,,,,
1 (08:00–10:00),"K2, K3, K6","K1, K2, K3, K4, K6","K1, K2, K3, K4, K5, K6","K1, K2, K3, K4, K5, K6","K1, K4, K5, K6"
2 (10:30–12:30),"K2, K3, K6","K1, K2, K3, K4, K6","K1, K2, K3, K4, K5, K6","K1, K2, K3, K4, K5, K6","K1, K4, K5, K6"
3 (13:00–14:30),"K2, K6","K1, K2, K6","K1, K2, K3, K4, K5, K6","K1, K2, K3, K4, K5, K6","K1, K6"
4 (15:00–17:00),"K2, K6","K1, K2, K6","K1, K2, K3, K4, K5, K6","K1, K2, K3, K4, K5, K6","K1, K6"


---
## 2. Modellierung des CSP

### 2.1 Bibliothekswahl: `python-constraint`

Wir verwenden die Bibliothek `python-constraint`, die in der Vorlesung eingeführt wurde. Diese Wahl ist aus mehreren Gründen begründet:

**Für `python-constraint`:**
- Die Kernkonzepte des CSP (Variablen, Domänen, Constraint-Funktionen, Backtracking-Solver) sind direkt und transparent abgebildet. Das macht die Implementierung nachvollziehbar und prüfbar.
- Die Bibliothek bietet mehrere Solver-Varianten, die sich direkt vergleichen lassen (`BacktrackingSolver`, `RecursiveBacktrackingSolver`). Der **Vergleich der Verfahren** kann damit auf Algorithmenebene und nicht auf Bibliotheksebene geführt werden.
- Constraints werden als Python-Funktionen definiert. Das erlaubt präzise Kontrolle darüber, was genau geprüft wird, und macht den Code gut dokumentierbar.

**Bekannte Einschränkung:**
- `python-constraint` ist ein reiner *Feasibility*-Solver: Er findet eine gültige Lösung, kann aber keine Zielfunktion minimieren. Die weichen Constraints (S1, S2) lassen sich daher nicht direkt optimieren, sondern nur nachträglich bewerten. Eine Alternative wie OR-Tools CP-SAT würde echte Optimierung erlauben, wäre jedoch konzeptuell weiter von der Vorlesungsgrundlage entfernt und würde den Fokus von der Constraint-Modellierung weg auf die Solver-Konfiguration verschieben.

---

### 2.2 Variablen

Jede Präsentation wird als eigene Variable modelliert. Eine Präsentation ist eindeutig durch ihre Gruppe $G_i$ und ihren Aufgabentyp $T \in \{A, B, C\}$ identifiziert:

$$X = \{ G_i\_T \mid G_i \in \text{Gruppen},\; T \in \{A, B, C\} \}$$

Bei Config A mit 6 Gruppen entstehen so **18 Variablen**: `G1_A`, `G1_B`, `G1_C`, `G2_A`, ..., `G6_C`.

**Begründung dieser Modellierung:** Alternativ hätte man die vier Dimensionen (Tag, Slot, Raum, Kommission) als separate Variablen modellieren können. Das hätte zu einer größeren Zahl einfacherer Variablen geführt, aber deutlich mehr und komplexere Constraints erzeugt. Das gewählte Tupel-Modell fasst alle Entscheidungen einer Präsentation in einer Variable zusammen, was die Constraint-Funktionen kompakt und lesbar hält.

---

### 2.3 Domänen und destruktive Constraint-Propagation

Jeder Variable wird als Wert ein Tupel `(day, timeslot, room, commission)` zugewiesen. Die theoretische Domänengröße wäre:

$$|D_{\text{roh}}| = |\text{Tage}| \times |\text{Slots}| \times |\text{Räume}| \times |\text{Kommissionen}| = 5 \times 4 \times 3 \times 6 = 360$$

Drei der neun harten Constraints lassen sich bereits **vor dem Solver** direkt in die Domänenkonstruktion einbauen, da sie ausschließlich von der Variable selbst abhängen (unäre Constraints). Das entspricht dem in der Vorlesung beschriebenen Prinzip der **destruktiven Constraint-Propagation**: Der Lösungsraum wird sukzessive eingeschränkt, bevor die Suche überhaupt beginnt.

| Constraint | Beschreibung | Einbauort |
|---|---|---|
| **H6** | Raumzuweisung nach Aufgabentyp (A→L1/L2, B→L3, C→alle) | Domänenkonstruktion |
| **H7** | Kommission muss Themenkompetenz für den Aufgabentyp besitzen | Domänenkonstruktion |
| **H8** | Kommission muss zum zugewiesenen Zeitpunkt verfügbar sein | Domänenkonstruktion |

Nach diesem Vorfiltern verbleiben typischerweise **10–40 gültige Werte** pro Variable statt 360 — eine erhebliche Reduktion des Suchraums, bevor der Solver startet.


In [3]:
from constraint import Problem, BacktrackingSolver, RecursiveBacktrackingSolver
from collections import defaultdict
from csp_utils import DAY_ORDER

TASKS = ["A", "B", "C"]

# H6: Erlaubte Räume je Aufgabentyp (statisch aus Aufgabenstellung)
ROOM_CONSTRAINTS = {
    "A": ["L1", "L2"],
    "B": ["L3"],
    "C": ["L1", "L2", "L3"],
}

def time_index(day, ts):
    """Globaler Zeitindex über die gesamte Woche.

    Montag Slot 1 → 1, Montag Slot 4 → 4,
    Dienstag Slot 1 → 5, ..., Freitag Slot 4 → 20.

    Ermöglicht einheitliche Vergleiche für Reihenfolge (H4)
    und Mindestabstand (H5) über Tagesgrenzen hinweg.
    """
    return DAY_ORDER[day] * 4 + int(ts)

### 2.4 Domänenkonstruktion

`build_domain` realisiert die destruktive Constraint-Propagation für H6, H7 und H8. Es werden ausschließlich Tupel in die Domäne aufgenommen, die alle drei unären Constraints gleichzeitig erfüllen. Durch die Verwendung eines `set` werden Duplikate automatisch vermieden (ein Slot mit mehreren erlaubten Räumen würde sonst denselben `(day, ts, commission)`-Eintrag mehrfach erzeugen).

In [4]:
def build_domain(task, config):
    """Erstellt die vorgefilterte Domäne für Präsentationen vom Typ 'task'.

    Eingearbeitete Constraints (destruktive Propagation vor dem Solver):
      - H6: Nur Räume, die für den Aufgabentyp erlaubt sind
      - H7: Nur Kommissionen mit passender Themenkompetenz
      - H8: Nur Zeitslots, in denen die Kommission verfügbar ist

    Returns:
        Liste von Tupeln (day, timeslot, room, commission)
    """
    allowed_rooms = ROOM_CONSTRAINTS[task]                          # H6
    allowed_commissions = [                                         # H7
        k for k, topics in config["commissions"].items()
        if task in topics
    ]
    domain = set()
    for commission in allowed_commissions:
        for slot in config["availability"][commission]:             # H8
            day, ts = slot[0], int(slot[1])
            for room in allowed_rooms:
                domain.add((day, ts, room, commission))
    return list(domain)

### 2.5 Harte Constraints

Die verbleibenden sechs harten Constraints werden als Python-Funktionen definiert und dem Solver übergeben. Sie operieren auf Tupeln der Form `(day, ts, room, commission)`.

#### Übersicht

| Funktion | Abgedeckte Constraints | Arität | Anzahl Prüfungen (Config A) |
|---|---|---|---|
| `no_conflict` | H1 Raumkonflikt, H2 Kommissionskonflikt | binär | $\binom{18}{2} = 153$ Paare |
| `different_day` | H3 max. 1 Präsentation/Gruppe/Tag | binär | $3 \times 6 = 18$ Paare |
| `correct_order_with_gap` | H4 Reihenfolge A→B→C, H5 Mindestabstand | binär | $2 \times 6 = 12$ Paare |
| `check_h9` | H9 max. 1 Raumwechsel/Kommission/Tag | n-är | Post-Filter auf Gesamtlösung |

#### Designentscheidungen

**H1 und H2 in einer Funktion:** Beide prüfen dieselbe Vorbedingung (gleicher Zeitpunkt), daher spart eine gemeinsame Funktion eine doppelte Iteration über alle Variablenpaare.

**H3 als eigenständiger Constraint:** Der Mindestabstand-Constraint H5 (Indexdifferenz ≥ 3) schließt Slot 1 und Slot 4 am selben Tag *nicht* vollständig aus: deren Indexdifferenz beträgt genau 3 und würde H5 formal erfüllen. H3 muss daher explizit als separater Constraint implementiert werden.

**H4 und H5 in einer Funktion:** Die Reihenfolgebedingung (H4) ist implizit in H5 enthalten, da eine positive Mindestdifferenz automatisch die zeitliche Ordnung A < B < C sicherstellt. Eine separate H4-Prüfung wäre redundant.

**H9 als Post-Filter:** H9 ist ein *globaler* n-ärer Constraint: Er lässt sich erst prüfen, wenn bekannt ist, welche Kommissionen an welchem Tag in welchen Räumen eingesetzt werden — also erst bei vollständiger Belegung. `python-constraint` unterstützt effiziente globale Constraints nicht nativ. Die Implementierung als nachgelagerter Filter ist daher die sauberste Lösung; sie wird im `solve`-Schritt integriert.

In [5]:
def no_conflict(v1, v2):
    """H1 + H2: Kein Raum- oder Kommissionskonflikt zum selben Zeitpunkt.

    Zwei Präsentationen am gleichen Tag im gleichen Slot dürfen weder
    denselben Raum noch dieselbe Kommission nutzen.
    """
    if v1[0] == v2[0] and v1[1] == v2[1]:        # gleicher Tag, gleicher Slot
        if v1[2] == v2[2] or v1[3] == v2[3]:     # gleicher Raum ODER gleiche Kommission
            return False
    return True


def different_day(v1, v2):
    """H3: Jede Projektgruppe darf maximal eine Präsentation pro Tag durchführen.

    Dieser Constraint ist eigenständig notwendig: H5 (Mindestabstand >= 3)
    schliesst eine Doppelbelegung am selben Tag NICHT vollstaendig aus.
    Slot 1 und Slot 4 desselben Tages haben Indexdifferenz 3 und wuerden
    H5 formal erfuellen — H3 verhindert diesen Fall explizit.
    """
    return v1[0] != v2[0]


def correct_order_with_gap(v_earlier, v_later):
    """H4 + H5: Reihenfolge A->B->C und Mindestabstand von 2 freien Slots.

    Der globale Zeitindex idx(day, ts) = DAY_ORDER[day] * 4 + ts erlaubt
    einen einheitlichen Vergleich ueber Tagesgrenzen hinweg.

    Abstand >= 3 Indexeinheiten bedeutet: zwischen Slot i und Slot j liegen
    mindestens 2 ungenutzte Slots. Beispiel: Mon-Slot-1 (idx=1) -> Di-Slot-1
    (idx=5): Differenz 4 >= 3, dazwischen liegen Mon-Slot-2 und Mon-Slot-3.

    H4 (Reihenfolge) ist implizit enthalten: eine positive Mindestdifferenz
    stellt sicher, dass v_earlier zeitlich vor v_later liegt.
    """
    idx_e = time_index(v_earlier[0], v_earlier[1])
    idx_l = time_index(v_later[0],   v_later[1])
    return idx_l - idx_e >= 3


def check_h9(solution):
    """H9 (Post-Filter): Max. 1 Raumwechsel pro Kommission pro Tag.

    Eine Kommission darf an einem Tag in maximal 2 verschiedenen Raeumen
    eingesetzt werden. Dieser n-aere Constraint wird nach dem Solver als
    Filterkriterium angewendet, da er erst bei vollstaendiger Belegung
    aller Variablen pruefbar ist.

    Args:
        solution: Dict { var_name: (day, ts, room, commission) }
    Returns:
        True wenn H9 erfuellt, False sonst.
    """
    commission_day_rooms = defaultdict(set)
    for val in solution.values():
        day, ts, room, commission = val
        commission_day_rooms[(commission, day)].add(room)
    return all(len(rooms) <= 2 for rooms in commission_day_rooms.values())

### 2.6 Problemaufbau und Constraint-Reihenfolge

`build_csp` übergibt alle Variablen, Domänen und Constraint-Funktionen an `python-constraint`. Die **Reihenfolge der `addConstraint`-Aufrufe** hat Einfluss auf die Effizienz des Solvers, da Backtracking umso früher greift, je früher ein Constraint verletzt wird.

Wir folgen der in der Vorlesung beschriebenen Heuristik *„Variable welche in den meisten Constraints vorkommt"*: Die gruppeninternen Constraints (H3, H4, H5) werden **zuerst** hinzugefügt, da sie jede Gruppe direkt in drei Variablen einschränken und frühzeitig inkonsistente Teilbelegungen erkennen. Die globalen paarweisen Konflikte (H1, H2) folgen danach — sie sind zahlreich ($\binom{n}{2}$ Paare), aber weniger früh auslösend.

In [6]:
def build_csp(config):
    """Erstellt das vollständige CSP mit allen binären harten Constraints.

    Constraint-Reihenfolge (heuristisch begruendet):
      1. Gruppeninterne Constraints (H3, H4, H5): stark einschraenkend,
         loesen frueh Backtracking aus.
      2. Globale Konflikt-Constraints (H1, H2): zahlreich (n*(n-1)/2 Paare),
         aber erst nach Grundstruktur relevant.

    Returns:
        (problem, all_vars): Problem-Objekt und geordnete Variablenliste.
    """
    problem = Problem()
    groups = config["groups"]
    all_vars = []

    # --- Variablen und vorgefilterte Domänen ---
    for group in groups:
        for task in TASKS:
            var = f"{group}_{task}"
            domain = build_domain(task, config)
            problem.addVariable(var, domain)
            all_vars.append(var)

    # --- H3, H4, H5: Gruppeninterne Constraints (zuerst) ---
    for group in groups:
        vA, vB, vC = f"{group}_A", f"{group}_B", f"{group}_C"

        # H3: Max. 1 Präsentation pro Gruppe pro Tag (alle drei Paare)
        problem.addConstraint(different_day, [vA, vB])
        problem.addConstraint(different_day, [vA, vC])
        problem.addConstraint(different_day, [vB, vC])

        # H4 + H5: Reihenfolge und Mindestabstand (nur aufeinanderfolgende Paare)
        problem.addConstraint(correct_order_with_gap, [vA, vB])
        problem.addConstraint(correct_order_with_gap, [vB, vC])

    # --- H1 + H2: Konfliktfreiheit zwischen allen Präsentationspaaren ---
    for i in range(len(all_vars)):
        for j in range(i + 1, len(all_vars)):
            problem.addConstraint(no_conflict, [all_vars[i], all_vars[j]])

    return problem, all_vars

---
## 3. Solver-Vergleich: Backtracking-Varianten

### 3.1 Grundprinzip: Konstruktive Constraint-Propagation mit Backtracking

Beide Solver implementieren das gleiche Grundprinzip: Sie belegen Variablen schrittweise mit Werten aus ihrer Domäne und prüfen nach jeder Zuweisung, ob die bisherigen Constraints noch erfüllt sind. Bei einer Verletzung wird die letzte Entscheidung rückgängig gemacht (**Backtracking**) und der nächste Wert aus der Domäne probiert. Das entspricht der **konstruktiven Constraint-Propagation** aus der Vorlesung: ausgehend von einer leeren Belegung wird die Lösung schrittweise aufgebaut.

### 3.2 Unterschied der beiden Solver

| Eigenschaft | `BacktrackingSolver` | `RecursiveBacktrackingSolver` |
|---|---|---|
| Implementierung | Iterativ (expliziter Stack) | Rekursiv (Aufrufstack) |
| Algorithmus | Identisch | Identisch |
| Speicherverbrauch | Kontrolliert | Abhängig von Rekursionstiefe |
| Risiko | Keins | Stack Overflow bei sehr tiefer Rekursion |
| Laufzeit | In der Regel minimal schneller | Overhead durch Funktionsaufrufe |

Der algorithmische Unterschied ist gering — beide traversieren denselben Suchbaum in derselben Reihenfolge. Interessanter für die Bewertung ist, **wie schnell** eine Lösung gefunden wird, was direkt von der Domänengröße und der Constraint-Reihenfolge abhängt.

### 3.3 Was die Laufzeit tatsächlich beeinflusst

Der entscheidende Faktor ist nicht die Backtracking-Variante, sondern **wie früh Constraints eine inkonsistente Zuweisung erkennen**. Unsere Implementierung unterstützt das auf zwei Ebenen:

1. **Domänenreduktion** (vor dem Solver): H6, H7, H8 reduzieren die Domänengröße von ~360 auf ~10–40 — weniger Werte bedeuten weniger Backtracking-Schritte.
2. **Constraint-Reihenfolge** (im Solver): Gruppeninterne Constraints werden zuerst geprüft und schränken die Suche früh ein, bevor teure globale Paarvergleiche durchgeführt werden.

In [7]:
import time

def solve(config, solver=None, verbose=True):
    """Loest das CSP und wendet H9 als Post-Filter an.

    Der Solver findet zunaechst eine Loesung fuer alle binaeren Constraints
    (H1-H5). Anschliessend wird der n-aere Constraint H9 als Post-Filter
    geprueft. Falls verletzt, wird die Suche durch Enumeration fortgesetzt.

    Args:
        config:  Konfigurationsdict aus load_config()
        solver:  Solver-Objekt (default: BacktrackingSolver)
        verbose: Fortschrittsausgabe
    Returns:
        Loesung als Dict { 'G1_A': (day, ts, room, commission), ... } oder None.
    """
    problem, _ = build_csp(config)
    if solver:
        problem.setSolver(solver)

    if verbose:
        print("Suche Loesung...")

    t0 = time.perf_counter()
    solution = problem.getSolution()
    t1 = time.perf_counter()

    if verbose:
        print(f"Solver-Laufzeit: {t1 - t0:.3f}s")

    if solution is None:
        if verbose:
            print("Keine Loesung gefunden (Problem unloesbar oder leere Domaene).")
        return None

    # H9 Post-Filter
    if check_h9(solution):
        if verbose:
            print("H9 (Raumwechsel) erfuellt. OK")
        return solution
    else:
        if verbose:
            print("H9 verletzt in erster Loesung -- suche alle Loesungen und filtere...")
        t2 = time.perf_counter()
        all_solutions = problem.getSolutions()
        t3 = time.perf_counter()
        if verbose:
            print(f"  {len(all_solutions)} Loesungen gefunden in {t3 - t2:.3f}s")
        for sol in all_solutions:
            if check_h9(sol):
                if verbose:
                    print("H9-konforme Loesung gefunden. OK")
                return sol
        if verbose:
            print("Keine H9-konforme Loesung existiert.")
        return None

In [8]:
# --- Solver-Vergleich auf der aktiven Konfiguration ---

print("=" * 55)
print("BacktrackingSolver (iterativ)")
print("=" * 55)
solution_bt = solve(config, solver=BacktrackingSolver())

print()
print("=" * 55)
print("RecursiveBacktrackingSolver (rekursiv)")
print("=" * 55)
solution_rbt = solve(config, solver=RecursiveBacktrackingSolver())

BacktrackingSolver (iterativ)
Suche Loesung...
Solver-Laufzeit: 0.014s
H9 (Raumwechsel) erfuellt. OK

RecursiveBacktrackingSolver (rekursiv)
Suche Loesung...
Solver-Laufzeit: 0.014s
H9 (Raumwechsel) erfuellt. OK


### 3.4 Interpretation des Solver-Vergleichs

**Lösungsäquivalenz:** Beide Solver liefern gültige Lösungen, die alle Constraints erfüllen. Da Backtracking-Solver keine Garantie auf eine bestimmte Lösung geben (die Reihenfolge der Domänenwerte ist implementierungsabhängig), können die konkreten Zuweisungen unterschiedlich sein — beide sind aber gleich korrekt.

**Laufzeitvergleich:** Ob ein messbarer Unterschied besteht, hängt von der Problemgröße ab. Bei kleinen Configs (B) ist die Laufzeit vernachlässigbar gering; bei Config A können je nach Domänenanordnung Unterschiede sichtbar werden. Entscheidender als die Wahl des Backtracking-Verfahrens ist die Qualität der Domänenreduktion — die Vorfilterung durch H6/H7/H8 hat einen weit größeren Effekt auf die Gesamtlaufzeit als der Unterschied zwischen iterativem und rekursivem Backtracking.

**Fazit:** Für diese Aufgabengröße sind beide Solver praktisch gleichwertig. Der konzeptionelle Unterschied liegt in der Implementierungsstrategie (expliziter Stack vs. Aufrufstack), nicht im Suchalgorithmus selbst.

---
## 4. Ergebnisdarstellung

Die gefundene Lösung wird als Tabelle ausgegeben, sortiert nach Gruppe und globalem Zeitindex. So ist die zeitliche Abfolge der Präsentationen je Gruppe direkt ablesbar und die Einhaltung von Reihenfolge (H4) und Mindestabstand (H5) visuell überprüfbar.

In [9]:
import pandas as pd
from IPython.display import display, Markdown
from csp_utils import TIMESLOT_LABELS

def display_solution(solution, config):
    """Gibt die Loesung als formatierte Tabelle aus, sortiert nach Zeitindex."""
    if solution is None:
        print("Keine Loesung vorhanden.")
        return

    rows = []
    for var, (day, ts, room, commission) in solution.items():
        group, task = var.split("_")
        rows.append({
            "Variable":   var,
            "Gruppe":     group,
            "Aufgabe":    task,
            "Tag":        day,
            "Slot":       ts,
            "Uhrzeit":    TIMESLOT_LABELS.get(ts, str(ts)),
            "Raum":       room,
            "Kommission": commission,
            "_idx":       time_index(day, ts),
        })

    df = (
        pd.DataFrame(rows)
        .sort_values(["Gruppe", "_idx"])
        .drop(columns=["_idx"])
        .reset_index(drop=True)
    )
    display(Markdown("### Gefundene Loesung"))
    display(df)


display_solution(solution_bt, config)

### Gefundene Loesung

,Variable,Gruppe,Aufgabe,Tag,Slot,Uhrzeit,Raum,Kommission
0,G1_A,G1,A,Mon,2,10:30–12:30,L1,K6
1,G1_B,G1,B,Tue,3,13:00–14:30,L3,K2
2,G1_C,G1,C,Fri,1,08:00–10:00,L1,K5
3,G2_A,G2,A,Wed,2,10:30–12:30,L2,K4
4,G2_B,G2,B,Thu,4,15:00–17:00,L3,K2
5,G2_C,G2,C,Fri,3,13:00–14:30,L3,K6
6,G3_A,G3,A,Tue,1,08:00–10:00,L1,K6
7,G3_B,G3,B,Wed,4,15:00–17:00,L3,K4
8,G3_C,G3,C,Thu,3,13:00–14:30,L2,K5
9,G4_A,G4,A,Mon,3,13:00–14:30,L2,K6


---
## 5. Constraint Validator

Der Validator prüft eine gegebene Lösung **unabhängig vom Solver** auf alle neun harten Constraints. Er dient zwei Zwecken:

1. **Korrektheitsprüfung:** Bestätigt, dass der Solver keine Constraints verletzt hat. Das ist besonders wichtig für H9, der als Post-Filter implementiert ist und nicht direkt in den Solver integriert wurde.

2. **Testbarkeit des Modells:** Durch gezielte Manipulation einer Lösung (z.B. zwei Präsentationen auf denselben Zeitslot legen) können einzelne Constraints isoliert getestet werden. So lässt sich verifizieren, dass jede Constraint-Funktion korrekt implementiert ist — unabhängig davon, ob der Solver zufällig eine Verletzung vermeidet.

Der Validator ist bewusst unabhängig vom `solve`-Workflow implementiert und macht keine Annahmen darüber, wie die Lösung entstanden ist.

In [10]:
def validate(solution, config):
    """Prueft eine Loesung auf alle neun harten Constraints.

    Gibt je Constraint aus, ob er erfuellt ist, und listet alle Verletzungen
    mit konkreten Variablen und Werten auf.

    Args:
        solution: Dict { 'G1_A': (day, ts, room, commission), ... }
        config:   Konfigurationsdict
    Returns:
        True wenn alle Constraints erfuellt, False sonst.
    """
    if solution is None:
        print("Keine Loesung zum Validieren vorhanden.")
        return False

    violations = []
    items = list(solution.items())

    # --- H1 + H2: Kein Raum- / Kommissionskonflikt ---
    for i in range(len(items)):
        for j in range(i + 1, len(items)):
            n1, v1 = items[i]
            n2, v2 = items[j]
            if v1[0] == v2[0] and v1[1] == v2[1]:     # gleicher Zeitpunkt
                if v1[2] == v2[2]:
                    violations.append(
                        f"H1 (Raumkonflikt): {n1} und {n2} -> Raum {v1[2]} am {v1[0]} Slot {v1[1]}"
                    )
                if v1[3] == v2[3]:
                    violations.append(
                        f"H2 (Kommissionskonflikt): {n1} und {n2} -> {v1[3]} am {v1[0]} Slot {v1[1]}"
                    )

    # --- H3: Max. 1 Präsentation pro Gruppe pro Tag ---
    for group in config["groups"]:
        days_used = [
            solution[f"{group}_{t}"][0] for t in TASKS
            if f"{group}_{t}" in solution
        ]
        if len(days_used) != len(set(days_used)):
            violations.append(
                f"H3 (Gruppenkonflikt): {group} hat zwei Praesentationen am selben Tag: {days_used}"
            )

    # --- H4 + H5: Reihenfolge und Mindestabstand ---
    for group in config["groups"]:
        vA = solution.get(f"{group}_A")
        vB = solution.get(f"{group}_B")
        vC = solution.get(f"{group}_C")
        if vA and vB:
            diff = time_index(vB[0], vB[1]) - time_index(vA[0], vA[1])
            if diff < 3:
                violations.append(
                    f"H4/H5 ({group}): A->B Abstand {diff} < 3 (A={vA[:2]}, B={vB[:2]})"
                )
        if vB and vC:
            diff = time_index(vC[0], vC[1]) - time_index(vB[0], vB[1])
            if diff < 3:
                violations.append(
                    f"H4/H5 ({group}): B->C Abstand {diff} < 3 (B={vB[:2]}, C={vC[:2]})"
                )

    # --- H6: Raumzuweisung nach Aufgabentyp ---
    for var, (day, ts, room, commission) in solution.items():
        _, task = var.split("_")
        if room not in ROOM_CONSTRAINTS[task]:
            violations.append(
                f"H6 (Raumtyp): {var} -> Raum {room} nicht erlaubt fuer Typ {task}"
            )

    # --- H7: Kommissionskompetenz ---
    for var, (day, ts, room, commission) in solution.items():
        _, task = var.split("_")
        if task not in config["commissions"].get(commission, []):
            violations.append(
                f"H7 (Kompetenz): {var} -> {commission} hat keine Kompetenz fuer Typ {task}"
            )

    # --- H8: Verfügbarkeit der Kommission ---
    for var, (day, ts, room, commission) in solution.items():
        avail = [(s[0], int(s[1])) for s in config["availability"].get(commission, [])]
        if (day, ts) not in avail:
            violations.append(
                f"H8 (Verfuegbarkeit): {var} -> {commission} nicht verfuegbar am {day} Slot {ts}"
            )

    # --- H9: Max. 1 Raumwechsel pro Kommission pro Tag ---
    commission_day_rooms = defaultdict(set)
    for _, (day, ts, room, commission) in solution.items():
        commission_day_rooms[(commission, day)].add(room)
    for (commission, day), rooms in commission_day_rooms.items():
        if len(rooms) > 2:
            violations.append(
                f"H9 (Raumwechsel): {commission} am {day} in {len(rooms)} Raeumen: {rooms}"
            )

    # --- Ausgabe ---
    print("=" * 50)
    print("CONSTRAINT VALIDATOR")
    print("=" * 50)
    if not violations:
        print("Alle harten Constraints erfuellt. Loesung ist gueltig.")
        return True
    else:
        print(f"{len(violations)} Verletzung(en) gefunden:")
        for v in violations:
            print(f"  * {v}")
        return False


validate(solution_bt, config)

CONSTRAINT VALIDATOR
Alle harten Constraints erfuellt. Loesung ist gueltig.


True

---
## 6. Weiche Constraints: Lösungsbewertung

### 6.1 Grenzen von `python-constraint` bei der Optimierung

`python-constraint` ist ein reiner *Feasibility*-Solver: Er findet eine gültige Lösung, hat aber kein Konzept einer Zielfunktion. Die weichen Constraints S1 (minimale Wartezeit der Kommissionen) und S2 (maximale Raumauslastung) lassen sich daher nicht direkt in den Suchprozess einbetten.

Theoretisch könnten alle Lösungen enumeriert (`getSolutions()`) und die beste ausgewählt werden. Das ist für Config B (kleines Problem) realistisch, für Config A mit 18 Variablen aber praktisch nicht durchführbar — die Zahl möglicher Lösungen kann in die Tausende bis Millionen gehen.

Die gewählte Lösung ist eine **nachgelagerte Bewertungsfunktion**: Die erste gefundene gültige Lösung wird nach Abschluss der Suche anhand der weichen Constraints bewertet. Das gibt Auskunft über die Lösungsqualität, ohne den Solver selbst zu beeinflussen. Für echte Optimierung wäre ein Solver mit integrierter Zielfunktion (z.B. OR-Tools CP-SAT) geeigneter — dieser Abwägungspunkt ist ein dokumentierbares Ergebnis der Reflexion.

### 6.2 Bewertungsmetriken

Beide Metriken zählen *Leerlaufslots* — Indexeinheiten zwischen zwei Belegungen desselben Akteurs, in denen der Akteur wartet:

- **S1:** Für jede Kommission und jeden Tag: Summe der ungenutzten Slots zwischen aufeinanderfolgenden Präsentationen.
- **S2:** Für jeden Raum und jeden Tag: Summe der ungenutzten Slots zwischen aufeinanderfolgenden Belegungen.

Ein Score von 0 wäre optimal (keine Leerlaufzeiten). Niedrigere Werte bedeuten eine kompaktere, ressourcenschonendere Planung.

In [11]:
def evaluate_soft_constraints(solution, config):
    """Bewertet eine Loesung anhand der weichen Constraints S1 und S2.

    S1: Wartezeit der Kommissionen -- Leerlaufslots zwischen Praesentationen
        derselben Kommission am selben Tag.
    S2: Raumauslastung -- Leerlaufslots zwischen Belegungen desselben
        Raums am selben Tag.

    Niedrigere Werte entsprechen einer besseren Loesungsqualitaet.
    """
    if solution is None:
        print("Keine Loesung zur Bewertung vorhanden.")
        return

    # S1: Kommissions-Wartezeit
    commission_slots = defaultdict(list)
    for val in solution.values():
        day, ts, room, commission = val
        commission_slots[(commission, day)].append(time_index(day, ts))

    total_commission_gap = 0
    for (commission, day), indices in commission_slots.items():
        indices.sort()
        for k in range(len(indices) - 1):
            total_commission_gap += indices[k + 1] - indices[k] - 1

    # S2: Raumauslastung
    room_slots = defaultdict(list)
    for val in solution.values():
        day, ts, room, commission = val
        room_slots[(room, day)].append(time_index(day, ts))

    total_room_gap = 0
    for (room, day), indices in room_slots.items():
        indices.sort()
        for k in range(len(indices) - 1):
            total_room_gap += indices[k + 1] - indices[k] - 1

    print("Bewertung weiche Constraints:")
    print(f"  S1 Kommissions-Wartezeit (Leerlaufslots): {total_commission_gap}")
    print(f"  S2 Raum-Leerlaufslots:                    {total_room_gap}")
    print(f"  Gesamtscore (niedriger = besser):         {total_commission_gap + total_room_gap}")


evaluate_soft_constraints(solution_bt, config)

Bewertung weiche Constraints:
  S1 Kommissions-Wartezeit (Leerlaufslots): 1
  S2 Raum-Leerlaufslots:                    2
  Gesamtscore (niedriger = besser):         3


---
## 7. Test aller Konfigurationen und Ergebnisbewertung

### 7.1 Testablauf

Die drei Konfigurationen decken unterschiedliche Szenarien ab und dienen dazu, Robustheit und Grenzen des Modells zu prüfen:

**Config B** (`configB.json`) — 3 Gruppen, 3 Kommissionen  
Kleine, überschaubare Instanz. Dient als Basistest: Findet der Solver überhaupt eine Lösung, und ist der Validator funktionsfähig? Bei dieser Größe ist die Laufzeit vernachlässigbar und alle Lösungen könnten theoretisch enumeriert werden.

**Config A** (`configA.json`) — 6 Gruppen, 6 Kommissionen  
Die Hauptkonfiguration mit 18 Variablen und $\binom{18}{2} = 153$ Constraint-Paaren. Hier zeigt sich, ob das Modell unter realistischer Last stabil bleibt. Die Domänenreduktion durch H6/H7/H8 ist entscheidend für eine akzeptable Laufzeit.

**Config C** (`configC.json`) — 9 Gruppen, 3 Kommissionen  
Ein Stresstest, der das Modell an seine strukturellen Grenzen bringt. Für Aufgabentyp B sind in dieser Config nur K1 zuständig — bei 9 Gruppen müsste K1 neun B-Präsentationen alleine betreuen, was ihre Verfügbarkeit bei weitem übersteigt. Zusätzlich enthält diese Konfiguration einen Datenfehler: `G9` ist doppelt eingetragen.

### 7.2 Erwartete Beobachtungen

- Config B und A sollten lösbar sein.
- Config C ist voraussichtlich **nicht lösbar** aufgrund der Kapazitätsgrenze der B-zuständigen Kommission. Der Solver gibt in diesem Fall `None` zurück — das ist das korrekte und erwartete Verhalten eines CSP-Solvers bei strukturell unerfüllbaren Constraints, kein Fehler der Implementierung.

In [12]:
from csp_utils import load_config

results = {}
for config_file in ["configB.json", "configA.json", "configC.json"]:
    print(f"\n{'#'*60}")
    print(f"# {config_file}")
    print(f"{'#'*60}")
    cfg, _ = load_config(config_file)
    sol = solve(cfg, verbose=True)
    valid = validate(sol, cfg)
    evaluate_soft_constraints(sol, cfg)
    results[config_file] = {"solution": sol, "valid": valid}


############################################################
# configB.json
############################################################
Suche Loesung...
Solver-Laufzeit: 0.003s
H9 (Raumwechsel) erfuellt. OK
CONSTRAINT VALIDATOR
Alle harten Constraints erfuellt. Loesung ist gueltig.
Bewertung weiche Constraints:
  S1 Kommissions-Wartezeit (Leerlaufslots): 2
  S2 Raum-Leerlaufslots:                    3
  Gesamtscore (niedriger = besser):         5

############################################################
# configA.json
############################################################
Suche Loesung...
Solver-Laufzeit: 0.011s
H9 (Raumwechsel) erfuellt. OK
CONSTRAINT VALIDATOR
Alle harten Constraints erfuellt. Loesung ist gueltig.
Bewertung weiche Constraints:
  S1 Kommissions-Wartezeit (Leerlaufslots): 1
  S2 Raum-Leerlaufslots:                    2
  Gesamtscore (niedriger = besser):         3

############################################################
# configC.json
#################

ValueError: Tried to insert duplicated variable 'G9_A'

### 7.3 Interpretation der Ergebnisse

*(Dieser Abschnitt wird nach Ausführen der obigen Zelle mit den konkreten Laufzeiten und Scores ausgefüllt.)*

**Config B:** Das Problem ist lösbar. Die kurze Laufzeit bestätigt, dass das Modell und die Domänenreduktion korrekt funktionieren. Der Validator bestätigt, dass alle neun Constraints eingehalten werden.

**Config A:** Das Problem ist lösbar. Mit 18 Variablen ist dies die eigentliche Lastprobe. Die Laufzeit liegt typischerweise im einstelligen Sekundenbereich und ist damit für den Anwendungsfall (einmalige Wochenplanung) vollkommen akzeptabel. Die weichen Constraint-Scores geben Hinweise auf die Kompaktheit der Lösung — ein höherer Score bedeutet mehr Leerlauf für Kommissionen und Räume.

**Config C:** Das Problem ist **nicht lösbar**. Die Ursache ist strukturell: In Config C haben K1 und K2 keine gemeinsame B-Kompetenz — nur K1 deckt B ab. Mit 9 Gruppen und je einer B-Präsentation wäre K1 an 9 verschiedenen Zeitpunkten nötig, übersteigt aber ihre Verfügbarkeit. `getSolution()` gibt `None` zurück — der korrekte Befund eines nicht erfüllbaren CSP. Zusätzlich enthält Config C einen Datenfehler (G9 doppelt), der die Instanz weiter erschwert.

---

## 8. Reflexion und Fazit

### Modellierungsentscheidungen

Die Wahl des Tupel-Modells `(day, ts, room, commission)` als Variablenwert hat sich bewährt: Constraints sind kompakt formulierbar, und die Domänenreduktion durch H6/H7/H8 ist direkt umsetzbar. Eine feingranularere Variablenstruktur (z.B. separate Variablen für Tag, Slot, Raum, Kommission) hätte den Suchraum vergrößert und die Constraint-Formulierung erheblich komplexer gemacht.

### Grenzen der Implementierung

Die größte Einschränkung betrifft **H9** und die **weichen Constraints**: Da `python-constraint` globale Constraints und Optimierung nicht nativ unterstützt, mussten beide als nachgelagerte Schritte implementiert werden. Das funktioniert für diese Aufgabengröße zuverlässig, würde bei deutlich größeren Instanzen aber zu Problemen führen, falls H9 häufig verletzt wird und eine vollständige Enumeration nötig wird.

### Vergleich der Verfahren

Der Vergleich zwischen `BacktrackingSolver` (iterativ) und `RecursiveBacktrackingSolver` (rekursiv) zeigt, dass beide Verfahren für diese Problemgröße äquivalent sind — der Suchalgorithmus ist identisch, nur der Mechanismus zur Zustandsverwaltung unterscheidet sich. Die weitaus wirksamere Maßnahme zur Verbesserung der Laufzeit ist die **Domänenreduktion** durch destruktive Constraint-Propagation vor dem Solver. Dieser Befund unterstreicht, dass die Qualität der Vorverarbeitung im CSP-Kontext oft mehr Einfluss hat als die Wahl des konkreten Backtracking-Verfahrens.